# Robustness + tail analysis of the late-fusion hybrid

1. **Robustness** — repeat (tune beta on val, report on held-out test) over 5 user splits; report mean +/- std of the lift, so +12.6% isn't one lucky split.
2. **Tail analysis** — where does audio help? Split each user's relevant items by train popularity and compare fused vs SASRec on head vs tail.

One SASRec is trained and reused for both. Accelerator → **GPU**, Internet On.

In [1]:
!pip install -q --no-cache-dir --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"
import time, numpy as np, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available())

from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings
data = build_sequences(size="50m", maxlen=200)
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
print(f"items={data.n_items:,} eval={len(data.eval_pos):,} coverage={content_mask.mean():.3f}")

CUDA: True


flat/50m/listens.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

embeddings.parquet:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

items=629,373 eval=4,549 coverage=0.946


In [3]:
from ymrec.models.sasrec import train_and_eval
t0 = time.time()
sasrec, _ = train_and_eval(data, d=64, n_blocks=2, n_heads=1, dropout=0.2,
                           epochs=120, batch_size=128, lr=1e-3, eval_every=60)
print(f"SASRec trained in {(time.time()-t0)/60:.1f} min")

epoch  60  loss=0.0643  NDCG@10=0.0587  NDCG@100=0.0828  Recall@100=0.1291
epoch 120  loss=0.0418  NDCG@10=0.0721  NDCG@100=0.1008  Recall@100=0.1548
SASRec trained in 16.9 min


In [4]:
# 1) Robustness over 5 user splits.
from ymrec.models.fusion import robustness
rob = robustness(sasrec, content_emb, data, seeds=(0, 1, 2, 3, 4))
s = rob["summary"]
print(f"\nSASRec  NDCG@10 = {s['sasrec_mean']:.4f} +/- {s['sasrec_std']:.4f}")
print(f"Fused   NDCG@10 = {s['fused_mean']:.4f} +/- {s['fused_std']:.4f}")
print(f"Lift            = {s['lift_mean_%']:+.1f}% +/- {s['lift_std_%']:.1f}%   (betas: {s['betas']})")

seed 0: beta=1.0   SASRec=0.0712 Fused=0.0776 lift=+9.1%
seed 1: beta=0.75  SASRec=0.0714 Fused=0.0802 lift=+12.3%
seed 2: beta=1.0   SASRec=0.0759 Fused=0.0809 lift=+6.6%
seed 3: beta=0.75  SASRec=0.0737 Fused=0.0804 lift=+9.1%
seed 4: beta=1.0   SASRec=0.0718 Fused=0.0801 lift=+11.5%

SASRec  NDCG@10 = 0.0728 +/- 0.0018
Fused   NDCG@10 = 0.0798 +/- 0.0011
Lift            = +9.7% +/- 2.0%   (betas: [1.0, 0.75, 1.0, 0.75, 1.0])


In [5]:
# 2) Where does content help? (tail = few train interactions)
from ymrec.models.fusion import tail_analysis
import pandas as pd
rows = tail_analysis(sasrec, content_emb, data, beta=0.75, thresholds=(5, 20, 100))
pd.DataFrame(rows)

tail<= 5     users=3069  SASRec=0.0134 Fused=0.0122 lift=-8.5%
head> 5      users=4444  SASRec=0.0691 Fused=0.0777 lift=+12.5%
tail<= 20    users=3853  SASRec=0.0279 Fused=0.0290 lift=+4.1%
head> 20     users=4256  SASRec=0.0605 Fused=0.0683 lift=+12.9%
tail<= 100   users=4361  SASRec=0.0471 Fused=0.0508 lift=+7.9%
head> 100    users=3714  SASRec=0.0467 Fused=0.0529 lift=+13.2%


,slice,users,sasrec_ndcg@10,fused_ndcg@10,lift_%
0,tail<= 5,3069,0.0134,0.0122,-8.5
1,head> 5,4444,0.0691,0.0777,12.5
2,tail<= 20,3853,0.0279,0.0290,4.1
3,head> 20,4256,0.0605,0.0683,12.9
4,tail<= 100,4361,0.0471,0.0508,7.9
5,head> 100,3714,0.0467,0.0529,13.2
